# Experiment 4: Binary Classification using Linear and Kernel-Based Models

```
experiment4_spambase_logreg_svm.py
====================================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 4: Binary Classification using Linear and Kernel-Based Models
(Logistic Regression and Support Vector Machine)

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                         -> generate_eda_summary()
    - Classification train/eval   -> train_evaluate_classification()
    - Classification metrics      -> classification_performance_metrics()
    - Global plot style           -> set_plot_style()

Dataset: Spambase (UCI ML Repository / Kaggle mirror), 4601 emails x 57
features + binary target (1 = spam, 0 = ham). Same dataset and column
naming as Experiment 2.
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, StratifiedKFold,
                                      cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET (same column naming as Experiment 2)

In [2]:
COLUMN_NAMES = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d",
    "word_freq_our", "word_freq_over", "word_freq_remove", "word_freq_internet",
    "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will",
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free",
    "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money",
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650",
    "word_freq_lab", "word_freq_labs", "word_freq_telnet", "word_freq_857",
    "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology",
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project",
    "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference",
    "char_freq_semicolon", "char_freq_paren", "char_freq_bracket",
    "char_freq_bang", "char_freq_dollar", "char_freq_pound",
    "capital_run_length_average", "capital_run_length_longest",
    "capital_run_length_total", "spam",
]

DATA_PATH = "spambase.csv"  # <-- upload this file to the notebook's working directory
raw = pd.read_csv(DATA_PATH)
raw.columns = COLUMN_NAMES
df = raw.copy()
print("Dataset shape:", df.shape)
print(df["spam"].value_counts())

Dataset shape: (4601, 58)
spam
0    2788
1    1813
Name: count, dtype: int64


## 2. HANDLE MISSING VALUES (none expected, verified defensively)

In [3]:
n_missing = int(df.isnull().sum().sum())
print(f"Missing cells: {n_missing}")
if n_missing > 0:
    df = df.fillna(df.median(numeric_only=True))

Missing cells: 0


## 3. EDA (reusable function from Experiment 1)

In [4]:
generate_eda_summary(
    df, target_col="spam", dataset_name="Spambase (Exp. 4)",
    save_path=f"{FIG_DIR}/eda_spambase.eps"
)
plt.close("all")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_spambase.eps (600 DPI, EPS)


## 4. TRAIN / TEST SPLIT + FEATURE STANDARDIZATION

In [5]:
X = df.drop(columns=["spam"]).values
y = df["spam"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def time_fit_predict(model, Xtr, Xte, ytr):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_pred = model.predict(Xte)
    pred_t = time.perf_counter() - t0
    return y_pred, train_t, pred_t

## 5. BASELINE LOGISTIC REGRESSION (via reusable train_evaluate_classification)

In [6]:
baseline_models = {"Logistic Regression (Baseline)": LogisticRegression(max_iter=2000)}
baseline_results_df, baseline_fitted = train_evaluate_classification(
    baseline_models, X_train_scaled, X_test_scaled, y_train, y_test, scale=False
)
print("\n=== Baseline Logistic Regression ===")
print(baseline_results_df)

--- Classification Metrics: Logistic Regression (Baseline) ---
  Accuracy  : 0.9294
  Precision : 0.9293
  Recall    : 0.9294
  F1-score  : 0.9293
  ROC-AUC   : 0.9702


=== Baseline Logistic Regression ===
                                Accuracy  Precision    Recall  F1-score  \
Model                                                                     
Logistic Regression (Baseline)  0.929425   0.929289  0.929425  0.929264   

                                 ROC-AUC  
Model                                     
Logistic Regression (Baseline)  0.970196  


## 6. LOGISTIC REGRESSION HYPERPARAMETER TUNING (GridSearchCV + RandomizedSearchCV)

In [7]:
logreg_param_grid = [
    {"penalty": ["l1", "l2"], "C": [0.01, 0.1, 1, 10, 100], "solver": ["liblinear"]},
    {"penalty": ["l1", "l2"], "C": [0.01, 0.1, 1, 10, 100], "solver": ["saga"]},
]

print("[LogReg] Starting GridSearchCV...", flush=True)
t0 = time.perf_counter()
logreg_grid = GridSearchCV(LogisticRegression(max_iter=1000), logreg_param_grid,
                            cv=cv_strategy, scoring="accuracy", n_jobs=1)
logreg_grid.fit(X_train_scaled, y_train)
logreg_grid_time = time.perf_counter() - t0
print(f"[LogReg] GridSearchCV done in {logreg_grid_time:.1f}s", flush=True)

print("[LogReg] Starting RandomizedSearchCV...", flush=True)
t0 = time.perf_counter()
logreg_random = RandomizedSearchCV(LogisticRegression(max_iter=1000), logreg_param_grid,
                                    cv=cv_strategy, scoring="accuracy", n_iter=10,
                                    random_state=RANDOM_STATE, n_jobs=1)
logreg_random.fit(X_train_scaled, y_train)
logreg_random_time = time.perf_counter() - t0
print(f"[LogReg] RandomizedSearchCV done in {logreg_random_time:.1f}s", flush=True)

logreg_tuning_summary = pd.DataFrame({
    "GridSearchCV": {
        "Best Parameters": str(logreg_grid.best_params_),
        "Best CV Accuracy": logreg_grid.best_score_,
        "Execution Time (s)": logreg_grid_time,
    },
    "RandomizedSearchCV": {
        "Best Parameters": str(logreg_random.best_params_),
        "Best CV Accuracy": logreg_random.best_score_,
        "Execution Time (s)": logreg_random_time,
    },
})
logreg_tuning_summary.to_csv(f"{RES_DIR}/logreg_tuning_summary.csv")
print("\n=== Logistic Regression Tuning Summary ===")
print(logreg_tuning_summary)

best_logreg = logreg_grid.best_estimator_
y_pred_lr, lr_train_t, lr_pred_t = time_fit_predict(best_logreg, X_train_scaled,
                                                      X_test_scaled, y_train)
y_proba_lr = best_logreg.predict_proba(X_test_scaled)
logreg_metrics = classification_performance_metrics(
    y_test, y_pred_lr, y_proba_lr, model_name="Logistic Regression (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_logreg.eps"
)
logreg_metrics["Training Time (s)"] = lr_train_t
print("\n=== Tuned Logistic Regression Performance ===")
print(logreg_metrics)

[LogReg] Starting GridSearchCV...


[LogReg] GridSearchCV done in 54.1s


[LogReg] Starting RandomizedSearchCV...


[LogReg] RandomizedSearchCV done in 28.4s



=== Logistic Regression Tuning Summary ===
                                                         GridSearchCV  \
Best Parameters     {'C': 100, 'penalty': 'l1', 'solver': 'libline...   
Best CV Accuracy                                             0.926359   
Execution Time (s)                                          54.071581   

                                                   RandomizedSearchCV  
Best Parameters     {'solver': 'liblinear', 'penalty': 'l1', 'C': ...  
Best CV Accuracy                                             0.926087  
Execution Time (s)                                          28.400498  


--- Classification Metrics: Logistic Regression (Tuned) ---
  Accuracy  : 0.9251
  Precision : 0.9250
  Recall    : 0.9251
  F1-score  : 0.9248
  ROC-AUC   : 0.9678


=== Tuned Logistic Regression Performance ===
{'Model': 'Logistic Regression (Tuned)', 'Accuracy': 0.9250814332247557, 'Precision': 0.9249657438518629, 'Recall': 0.9250814332247557, 'F1-score': 0.9248297741033644, 'ROC-AUC': 0.9677567463491217, 'Training Time (s)': 0.7578961830000139}


## 7. SVM: TRAIN WITH DIFFERENT KERNELS (baseline, default hyperparameters)

In [8]:
kernels = ["linear", "poly", "rbf", "sigmoid"]
svm_kernel_rows = []
svm_kernel_models = {}
for kernel in kernels:
    print(f"[SVM baseline] Training kernel={kernel}...", flush=True)
    svm = SVC(kernel=kernel, probability=False, random_state=RANDOM_STATE)
    y_pred, train_t, pred_t = time_fit_predict(svm, X_train_scaled, X_test_scaled, y_train)
    svm_kernel_rows.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Training Time (s)": train_t,
    })
    svm_kernel_models[kernel] = svm

svm_kernel_df = pd.DataFrame(svm_kernel_rows).set_index("Kernel")
svm_kernel_df.to_csv(f"{RES_DIR}/svm_kernel_comparison.csv")
print("\n=== SVM Kernel-wise Performance ===")
print(svm_kernel_df)

best_kernel = svm_kernel_df["Accuracy"].idxmax()
print(f"\nBest SVM kernel (baseline hyperparameters): {best_kernel}")

fig, ax = plt.subplots(figsize=(8, 5))
svm_kernel_df["Accuracy"].plot(kind="bar", ax=ax, color="#16a085")
_bold_axis_labels(ax, "Kernel", "Accuracy", "SVM Kernel Comparison (Accuracy)")
plt.xticks(rotation=0)
_save_eps(fig, f"{FIG_DIR}/svm_kernel_accuracy.eps")
plt.close(fig)

[SVM baseline] Training kernel=linear...


[SVM baseline] Training kernel=poly...


[SVM baseline] Training kernel=rbf...


[SVM baseline] Training kernel=sigmoid...



=== SVM Kernel-wise Performance ===
         Accuracy  F1 Score  Training Time (s)
Kernel                                        
linear   0.929425  0.909344           0.300942
poly     0.779587  0.621974           0.245502
rbf      0.927253  0.905501           0.172676
sigmoid  0.884908  0.852778           0.214689

Best SVM kernel (baseline hyperparameters): linear


## 8. SVM HYPERPARAMETER TUNING (GridSearchCV + RandomizedSearchCV)

In [9]:
svm_param_grid = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"]},
    {"kernel": ["sigmoid"], "C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"]},
    {"kernel": ["poly"], "C": [0.1, 1, 10], "gamma": ["scale"],
     "degree": [2, 3]},
]

print("[SVM] Starting GridSearchCV...", flush=True)
t0 = time.perf_counter()
svm_grid = GridSearchCV(SVC(probability=False, random_state=RANDOM_STATE), svm_param_grid,
                         cv=cv_strategy, scoring="accuracy", n_jobs=1)
svm_grid.fit(X_train_scaled, y_train)
svm_grid_time = time.perf_counter() - t0
print(f"[SVM] GridSearchCV done in {svm_grid_time:.1f}s", flush=True)

print("[SVM] Starting RandomizedSearchCV...", flush=True)
t0 = time.perf_counter()
svm_random = RandomizedSearchCV(SVC(probability=False, random_state=RANDOM_STATE), svm_param_grid,
                                 cv=cv_strategy, scoring="accuracy", n_iter=15,
                                 random_state=RANDOM_STATE, n_jobs=1)
svm_random.fit(X_train_scaled, y_train)
svm_random_time = time.perf_counter() - t0
print(f"[SVM] RandomizedSearchCV done in {svm_random_time:.1f}s", flush=True)

svm_tuning_summary = pd.DataFrame({
    "GridSearchCV": {
        "Best Parameters": str(svm_grid.best_params_),
        "Best CV Accuracy": svm_grid.best_score_,
        "Execution Time (s)": svm_grid_time,
    },
    "RandomizedSearchCV": {
        "Best Parameters": str(svm_random.best_params_),
        "Best CV Accuracy": svm_random.best_score_,
        "Execution Time (s)": svm_random_time,
    },
})
svm_tuning_summary.to_csv(f"{RES_DIR}/svm_tuning_summary.csv")
print("\n=== SVM Tuning Summary ===")
print(svm_tuning_summary)

best_svm_params = svm_grid.best_params_
best_svm = SVC(probability=True, random_state=RANDOM_STATE, **best_svm_params)
y_pred_svm, svm_train_t, svm_pred_t = time_fit_predict(best_svm, X_train_scaled,
                                                         X_test_scaled, y_train)
y_proba_svm = best_svm.predict_proba(X_test_scaled)
svm_metrics = classification_performance_metrics(
    y_test, y_pred_svm, y_proba_svm, model_name="SVM (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_svm.eps"
)
svm_metrics["Training Time (s)"] = svm_train_t
print("\n=== Tuned SVM Performance ===")
print(svm_metrics)

[SVM] Starting GridSearchCV...


[SVM] GridSearchCV done in 75.1s


[SVM] Starting RandomizedSearchCV...


[SVM] RandomizedSearchCV done in 68.3s



=== SVM Tuning Summary ===
                                                    GridSearchCV  \
Best Parameters     {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}   
Best CV Accuracy                                        0.935598   
Execution Time (s)                                     75.074435   

                                              RandomizedSearchCV  
Best Parameters     {'kernel': 'rbf', 'gamma': 'scale', 'C': 10}  
Best CV Accuracy                                        0.935598  
Execution Time (s)                                     68.292419  


--- Classification Metrics: SVM (Tuned) ---
  Accuracy  : 0.9207
  Precision : 0.9206
  Recall    : 0.9207
  F1-score  : 0.9205
  ROC-AUC   : 0.9702


=== Tuned SVM Performance ===
{'Model': 'SVM (Tuned)', 'Accuracy': 0.9207383279044516, 'Precision': 0.9205914207642703, 'Recall': 0.9207383279044516, 'F1-score': 0.9204720798484869, 'ROC-AUC': 0.9701807912951609, 'Training Time (s)': 0.8967338850000033}


## 9. HYPERPARAMETER TUNING RESULTS TABLE (combined)

In [10]:
tuning_results_combined = pd.DataFrame({
    "Logistic Regression": {
        "Search Method": "Grid / Random",
        "Best Parameters": str(logreg_grid.best_params_),
        "Best CV Accuracy": logreg_grid.best_score_,
    },
    "SVM": {
        "Search Method": "Grid / Random",
        "Best Parameters": str(svm_grid.best_params_),
        "Best CV Accuracy": svm_grid.best_score_,
    },
}).T
tuning_results_combined.to_csv(f"{RES_DIR}/hyperparameter_tuning_results.csv")
print("\n=== Combined Hyperparameter Tuning Results ===")
print(tuning_results_combined)


=== Combined Hyperparameter Tuning Results ===
                     Search Method  \
Logistic Regression  Grid / Random   
SVM                  Grid / Random   

                                                       Best Parameters  \
Logistic Regression  {'C': 100, 'penalty': 'l1', 'solver': 'libline...   
SVM                       {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}   

                    Best CV Accuracy  
Logistic Regression         0.926359  
SVM                         0.935598  


## 10. 5-FOLD CROSS-VALIDATION (Logistic Regression vs SVM, best tuned configs)

In [11]:
X_scaled_full = scaler.fit_transform(X)

cv_lr = cross_validate(best_logreg, X_scaled_full, y, cv=cv_strategy, scoring="accuracy")
cv_svm = cross_validate(best_svm, X_scaled_full, y, cv=cv_strategy, scoring="accuracy")

cv_table = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)] + ["Average"],
    "Logistic Regression": list(cv_lr["test_score"]) + [cv_lr["test_score"].mean()],
    "SVM": list(cv_svm["test_score"]) + [cv_svm["test_score"].mean()],
}).set_index("Fold")
cv_table.to_csv(f"{RES_DIR}/cross_validation_results.csv")
print("\n=== 5-Fold Cross-Validation Results ===")
print(cv_table)

fig, ax = plt.subplots(figsize=(7, 5))
folds = np.arange(1, 6)
ax.plot(folds, cv_lr["test_score"], marker="o", label="Logistic Regression", color="#2980b9")
ax.plot(folds, cv_svm["test_score"], marker="s", label="SVM", color="#c0392b")
ax.legend()
_bold_axis_labels(ax, "Fold", "Accuracy", "Cross-Validation Accuracy")
_save_eps(fig, f"{FIG_DIR}/cv_accuracy.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== 5-Fold Cross-Validation Results ===
         Logistic Regression       SVM
Fold                                  
Fold 1              0.912052  0.930510
Fold 2              0.933696  0.929348
Fold 3              0.925000  0.941304
Fold 4              0.928261  0.932609
Fold 5              0.926087  0.936957
Average             0.925019  0.934146


## 11. ROC CURVES AND COMPARATIVE ANALYSIS

In [12]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in [("Logistic Regression (Tuned)", best_logreg), ("SVM (Tuned)", best_svm)]:
    proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.8)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.legend()
_bold_axis_labels(ax, "False Positive Rate", "True Positive Rate", "ROC Curves")
_save_eps(fig, f"{FIG_DIR}/roc_curves.eps")
plt.close(fig)

comparison_summary = pd.DataFrame([logreg_metrics, svm_metrics]).set_index("Model")
comparison_summary.to_csv(f"{RES_DIR}/comparative_analysis.csv")
print("\n=== Comparative Analysis ===")
print(comparison_summary)

fig, ax = plt.subplots(figsize=(8, 5))
comparison_summary[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", ax=ax)
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=11))
_bold_axis_labels(ax, "Model", "Score", "Logistic Regression vs SVM: Metric Comparison")
plt.xticks(rotation=15, ha="right")
_save_eps(fig, f"{FIG_DIR}/model_comparison_bar.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Comparative Analysis ===
                             Accuracy  Precision    Recall  F1-score  \
Model                                                                  
Logistic Regression (Tuned)  0.925081   0.924966  0.925081  0.924830   
SVM (Tuned)                  0.920738   0.920591  0.920738  0.920472   

                              ROC-AUC  Training Time (s)  
Model                                                     
Logistic Regression (Tuned)  0.967757           0.757896  
SVM (Tuned)                  0.970181           0.896734  


## 12. REGULARIZATION EFFECT (Logistic Regression: accuracy vs C)

In [13]:
C_values = [0.01, 0.1, 1, 10, 100]
reg_effect_rows = []
for C in C_values:
    for penalty in ["l1", "l2"]:
        model = LogisticRegression(C=C, penalty=penalty, solver="liblinear", max_iter=3000)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        n_nonzero = int(np.sum(model.coef_[0] != 0))
        reg_effect_rows.append({
            "C": C, "Penalty": penalty,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Non-zero Coefficients": n_nonzero,
        })
reg_effect_df = pd.DataFrame(reg_effect_rows)
reg_effect_df.to_csv(f"{RES_DIR}/regularization_effect.csv", index=False)
print("\n=== Regularization Effect (Logistic Regression) ===")
print(reg_effect_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for penalty, color in [("l1", "#8e44ad"), ("l2", "#2980b9")]:
    subset = reg_effect_df[reg_effect_df["Penalty"] == penalty]
    axes[0].plot(subset["C"], subset["Accuracy"], marker="o", label=penalty.upper(), color=color)
    axes[1].plot(subset["C"], subset["Non-zero Coefficients"], marker="o",
                 label=penalty.upper(), color=color)
axes[0].set_xscale("log")
axes[1].set_xscale("log")
axes[0].legend()
axes[1].legend()
_bold_axis_labels(axes[0], "C (log scale)", "Accuracy", "Accuracy vs C")
_bold_axis_labels(axes[1], "C (log scale)", "Non-zero Coefficients", "Sparsity vs C")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/regularization_effect.eps")
plt.close(fig)


=== Regularization Effect (Logistic Regression) ===
        C Penalty  Accuracy  Non-zero Coefficients
0    0.01      l1  0.869707                     26
1    0.01      l2  0.907709                     57
2    0.10      l1  0.926167                     49
3    0.10      l2  0.926167                     57
4    1.00      l1  0.927253                     57
5    1.00      l2  0.929425                     57
6   10.00      l1  0.926167                     57
7   10.00      l2  0.926167                     57
8  100.00      l1  0.925081                     57
9  100.00      l2  0.926167                     57


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 13. SVM DECISION BOUNDARY VISUALIZATION (2D PCA projection, all 4 kernels)

In [14]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_scaled)

xx, yy = np.meshgrid(np.linspace(X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1, 200),
                      np.linspace(X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1, 200))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, kernel in zip(axes, kernels):
    svm_2d = SVC(kernel=kernel, random_state=RANDOM_STATE)
    svm_2d.fit(X_train_2d, y_train)
    Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap="coolwarm",
               s=8, edgecolors="k", linewidths=0.2)
    _bold_axis_labels(ax, "PC1", "PC2", f"{kernel.capitalize()} Kernel (2D PCA)")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/svm_decision_boundaries_pca.eps")
plt.close(fig)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



All figures saved under: /home/claude/notebooks/figures
All result tables saved under: /home/claude/notebooks/results

Done.
